In [1]:
import warnings
warnings.filterwarnings(action='ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
import seaborn as sns
import sklearn
from sklearn import datasets

# 데이터 전처리

우리가 실제로 접하게 되는 데이터는 사이킷런 라이브러리에서 제공하는 데이터와는 다르게 머신러닝 또는 딥러닝에 적용하기 전에 데이터를 가공하는 경우가 대부분이다. 이를 데이터 전처리 또는 피처 엔지니어링 등으로 부른다.

## 결측치(missing value) 처리

데이터셋의 일부에 데이터가 존재하지 않을 경우 결측치라 부르며, 머신러닝 또는 딥러닝에 적용하기 전에 결측치를 처리하는 것이 좋다.

데이터프레임에 데이터가 존재하지 않는 결측치는 없지만 부적절한 값(이상치)이 존재한다.  
나이에 1000이라는 값이 존재하는 것과 나머치 피처에 'unknown'이라는 값이 입력된 부적절한 값은 결측치로 처리하는 것이 좋다.

In [2]:
df = pd.DataFrame([
    [42, '남', 12, '등산', 'class2'],
    [35, 'unknown', 3, '바둑', 'class1'],
    [1000, '여', 7, '낚시', 'class3'],
    [1000, 'unknown', 21, 'unknown', 'unknown'],
])

df.columns = ['나이', '성별', '태어난달', '취미', 'label']
df

,나이,성별,태어난달,취미,label
0,42,남,12,등산,class2
1,35,unknown,3,바둑,class1
2,1000,여,7,낚시,class3
3,1000,unknown,21,unknown,unknown


판다스의 unique() 메소드로 각 시리즈의 유일한 값을 확인할 수 있다.

In [4]:
print(f'나이: {df.나이.unique()}')
print(f'성별: {df.성별.unique()}')
print(f'태어난달: {df.태어난달.unique()}')
print(f'취미: {df.취미.unique()}')
print(f'label: {df.label.unique()}')

나이: [  42   35 1000]
성별: <StringArray>
['남', 'unknown', '여']
Length: 3, dtype: str
태어난달: [12  3  7 21]
취미: <StringArray>
['등산', '바둑', '낚시', 'unknown']
Length: 4, dtype: str
label: <StringArray>
['class2', 'class1', 'class3', 'unknown']
Length: 4, dtype: str


부적절한 데이터 값을 결측치로 바꾸기 위해서 np.nan를 사용한다.

In [7]:
df.loc[df.나이 > 150, '나이'] = np.nan
df.loc[df.성별 == 'unknown', '성별'] = np.nan
df.loc[df.태어난달 > 12, '태어난달'] = np.nan
df.loc[df.취미 == 'unknown', '취미'] = np.nan
df.loc[df.label == 'unknown', 'label'] = np.nan
df

,나이,성별,태어난달,취미,label
0,42.0,남,12.0,등산,class2
1,35.0,NaN,3.0,바둑,class1
2,NaN,여,7.0,낚시,class3
3,NaN,NaN,NaN,NaN,NaN


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   나이      2 non-null      float64
 1   성별      2 non-null      str    
 2   태어난달    3 non-null      float64
 3   취미      3 non-null      str    
 4   label   3 non-null      str    
dtypes: float64(2), str(3)
memory usage: 292.0 bytes


In [9]:
df.isnull().sum()

나이       2
성별       2
태어난달     1
취미       1
label    1
dtype: int64

결측치를 처리하는 데 주로 사용되는 방법은 결측치를 삭제하거나 다른 값으로 변경하는 방법이 있다.

dropna() 메소드로 결측치를 삭제할 수 있다.  
`axis`로 결측치를 생략할 행 또는 열 방향을 지정할 수 있다. 생략하거나 0으로 지정하면 결측치를 한 개라도 포함한 모든 행을 삭제하고, 1로 지정하면 결측치가 한 개라도 포함된 모든 열을 삭제한다.

In [11]:
df.dropna(axis=0)

,나이,성별,태어난달,취미,label
0,42.0,남,12.0,등산,class2


In [12]:
df.dropna(axis=1)

""
0
1
2
3


`how`를 `all`로 지정하면 모든 값이 결측치인 행만 제거한다.

In [13]:
df.dropna(how='all')

,나이,성별,태어난달,취미,label
0,42.0,남,12.0,등산,class2
1,35.0,NaN,3.0,바둑,class1
2,NaN,여,7.0,낚시,class3


`thresh`에 숫자를 지정하면 결측치를 제외한 값의 개수가 지정한 숫자보다 작은 모든 행을 삭제한다.

In [18]:
df.dropna(thresh=4)

,나이,성별,태어난달,취미,label
0,42.0,남,12.0,등산,class2
1,35.0,NaN,3.0,바둑,class1
2,NaN,여,7.0,낚시,class3


`subset`에 열이름을 지정하면 지정한 열에 결측치가 존재하는 모든 행을 삭제한다.

In [19]:
df.dropna(subset=['성별'])

,나이,성별,태어난달,취미,label
0,42.0,남,12.0,등산,class2
2,NaN,여,7.0,낚시,class3


fillna() 메소드로 결측치를 대체할 값을 지정할 수 있다.

In [20]:
df.fillna(0)

,나이,성별,태어난달,취미,label
0,42.0,남,12.0,등산,class2
1,35.0,0,3.0,바둑,class1
2,0.0,여,7.0,낚시,class3
3,0.0,0,0.0,0,0


`value`에 딕셔너리 형태로 각 열의 결측치를 대체할 값을 지정할 수 있다.

In [22]:
alter_values = {'나이': 0, '성별': 'U', '태어난달': 0, '취미': 'N', 'label': 'class4'}
df.fillna(value=alter_values, inplace=True)

,나이,성별,태어난달,취미,label
0,42.0,남,12.0,등산,class2
1,35.0,U,3.0,바둑,class1
2,0.0,여,7.0,낚시,class3
3,0.0,U,0.0,N,class4


## 기계 처리에 적합하도록 레이블을 설정한다.

label 열을 보면 문자열 형태인데, 이는 기계 처리에 적합하지 않기때문에 문자열을 숫자로 바꿔준다.

사이킷런 라이브러리를 사용하지 않고 레이블 설정하기

In [37]:
# 라벨링할 레이블을 얻어와서 오름차순으로 정렬한다.
label = list(df.label.values)
print(label)
label.sort()
# label.sort(reverse=True) # 내림차순 정렬
print(label)

['class2', 'class1', 'class3', 'class4']
['class1', 'class2', 'class3', 'class4']


In [39]:
# 반복문을 실행해서 정렬된 레이블을 key로 인덱스를 value로 하는 딕셔너리를 만든다.
label_dict = {}
for index, data in enumerate(label):
    # print(index, data)
    label_dict[data] = index
label_dict

{'class1': 0, 'class2': 1, 'class3': 2, 'class4': 3}

In [40]:
# replace() 메소드로 딕셔너리에 저장된 데이터로 레이블을 라벨링한다.
df['label2'] = df.label.replace(label_dict)
df

,나이,성별,태어난달,취미,label,label2
0,42.0,남,12.0,등산,class2,1
1,35.0,U,3.0,바둑,class1,0
2,0.0,여,7.0,낚시,class3,2
3,0.0,U,0.0,N,class4,3
